In [2]:
# Cell 1 - Imports & Config (Multi-class with localizers)

import os, sys, math, random, json
from glob import glob
from collections import defaultdict, Counter
from ast import literal_eval

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from skimage.transform import resize as sk_resize
from scipy.ndimage import gaussian_filter

from tqdm.auto import tqdm

# ---------- Paths (your dataset) ----------
ROOT = "/kaggle/input/rsna-intracranial-aneurysm-detection"
FILTERED_CSV = "/kaggle/input/rsna-filtered-set/train_masked.csv"   # filtered; must have SeriesInstanceUID
LOCALIZERS_CSV = os.path.join(ROOT, "train_localizers.csv")         # coordinates + location per slice
SEG_FOLDER = os.path.join(ROOT, "segmentations")                    # optional ground-truth NIfTI
SERIES_FOLDER = os.path.join(ROOT, "series")

# Outputs
OUTPUT_DIR = "/kaggle/working/patient_visuals"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Training config (tune as needed) ----------
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_PATIENTS = 40                 # start small; scale up once stable
PATCH_SIZE = (64, 128, 128)     # (D, H, W) — depth small for VRAM
BATCH_SIZE = 2
NUM_EPOCHS = 3
LR = 1e-4
NUM_WORKERS = 0                 # 0 on Kaggle to reduce RAM pressure

# Multi-class (background=0 + arteries)
LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
]
NUM_CLASSES = len(LABEL_COLS) + 1  # +1 for background (class 0)

# Repro
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device, "| PATCH_SIZE:", PATCH_SIZE, "| BATCH_SIZE:", BATCH_SIZE, "| NUM_CLASSES:", NUM_CLASSES)


Device: cuda | PATCH_SIZE: (64, 128, 128) | BATCH_SIZE: 2 | NUM_CLASSES: 14


In [3]:
# Cell 2 - Utilities: DICOM/NIfTI loading, localizer synthesis (multi-class)

def load_dicom_series(series_folder):
    """Load a DICOM series -> (D,H,W) float32; sorted by z (ImagePositionPatient) or InstanceNumber."""
    files = [os.path.join(series_folder, f) for f in os.listdir(series_folder) if f.lower().endswith(".dcm")]
    if len(files) == 0:
        raise FileNotFoundError("No DICOMs in " + series_folder)

    slices_meta = []
    for fp in files:
        try:
            ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
            z = float(ds.ImagePositionPatient[2]) if hasattr(ds, "ImagePositionPatient") else float(getattr(ds, "InstanceNumber", 0))
            sop = getattr(ds, "SOPInstanceUID", None)
            slices_meta.append((z, sop, fp))
        except Exception:
            continue
    slices_meta.sort(key=lambda x: x[0])

    imgs = []
    for _, _, fp in slices_meta:
        ds = pydicom.dcmread(fp, force=True)
        arr = ds.pixel_array.astype(np.float32)
        imgs.append(arr)
    vol = np.stack(imgs)  # (D,H,W)

    # normalize per volume
    mn, mx = float(vol.min()), float(vol.max())
    vol = (vol - mn) / (mx - mn + 1e-9)
    return vol

def load_nifti_volume(nii_path):
    """Load NIfTI -> (D,H,W) float32; handle common axis permutations."""
    img = nib.load(nii_path)
    data = img.get_fdata().astype(np.float32)
    # Assume we want (D,H,W)
    # If one axis is 'depth' but not first, permute; heuristic:
    if data.ndim == 3:
        # try to detect the shallowest dim as depth already first
        if data.shape[0] <= min(data.shape[1], data.shape[2]):
            return data  # likely (D,H,W)
        else:
            return np.transpose(data, (2, 1, 0))
    return np.transpose(data, (2, 1, 0))

def find_mask_path(series_uid):
    """Prefer _cowseg.nii then <uid>.nii."""
    p1 = os.path.join(SEG_FOLDER, f"{series_uid}_cowseg.nii")
    p2 = os.path.join(SEG_FOLDER, f"{series_uid}.nii")
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

# Localizers CSV
if os.path.exists(LOCALIZERS_CSV):
    localizer_df = pd.read_csv(LOCALIZERS_CSV)
else:
    localizer_df = pd.DataFrame(columns=["SeriesInstanceUID","SOPInstanceUID","coordinates","location"])

# Map artery name -> class id
LABEL2ID = {name: i+1 for i, name in enumerate(LABEL_COLS)}

def parse_xy(coord):
    """coordinates column can be a dict-like string: {'x': ..., 'y': ...}."""
    if isinstance(coord, str):
        try:
            d = literal_eval(coord)
            return int(round(float(d["x"]))), int(round(float(d["y"])))
        except Exception:
            pass
    if isinstance(coord, dict):
        return int(round(float(coord["x"]))), int(round(float(coord["y"])))
    return None, None

def synthesize_mask_from_localizer(series_uid, vol_shape, radius=5, smooth_sigma=0.8):
    """
    Build a multi-class mask from localizers for a given series.
    - vol_shape: (D,H,W)
    - For each row: (SOPInstanceUID, coordinates{x,y}, location) -> place small disk on that slice with class id.
    """
    mask = np.zeros(vol_shape, dtype=np.uint8)
    rows = localizer_df[localizer_df["SeriesInstanceUID"] == series_uid]
    if rows.empty:
        return mask

    # Build SOPInstanceUID -> slice index map from the actual DICOM order
    series_folder = os.path.join(SERIES_FOLDER, series_uid)
    if not os.path.exists(series_folder):
        return mask

    slices_meta = []
    for f in os.listdir(series_folder):
        if not f.lower().endswith(".dcm"): continue
        path = os.path.join(series_folder, f)
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            z = float(ds.ImagePositionPatient[2]) if hasattr(ds, "ImagePositionPatient") else float(getattr(ds, "InstanceNumber", 0))
            sop = getattr(ds, "SOPInstanceUID", None)
            slices_meta.append((z, sop))
        except Exception:
            continue
    slices_meta.sort(key=lambda x: x[0])
    sop2idx = {sop: idx for idx, (_, sop) in enumerate(slices_meta) if sop is not None}

    D, H, W = vol_shape
    yy, xx = np.ogrid[:H, :W]

    for _, r in rows.iterrows():
        sop = r.get("SOPInstanceUID", None)
        x, y = parse_xy(r.get("coordinates", None))
        loc = r.get("location", "")
        cls = LABEL2ID.get(loc, len(LABEL_COLS))  # "other posterior..." or unknown -> last class id
        if sop not in sop2idx or x is None or y is None: 
            continue
        zidx = sop2idx[sop]
        rr = max(1, int(radius))
        disk = ((yy - y)**2 + (xx - x)**2) <= rr*rr
        # Assign class, overwriting background (or weaker labels)
        slice_mask = mask[zidx]
        slice_mask[disk] = cls
        mask[zidx] = slice_mask

    # Light blur+threshold to avoid single-pixel islands and make training friendlier
    if smooth_sigma > 0:
        for c in range(1, NUM_CLASSES):
            m = (mask == c).astype(np.float32)
            if m.sum() == 0: 
                continue
            m = gaussian_filter(m, sigma=smooth_sigma)
            mask[m > 0.15] = c
    return mask


In [5]:
# Cell 3 - Dataset: cached volumes, positive patch sampling, light augs

class VolumePatchDataset(Dataset):
    def __init__(self, df, n_patients=None, patch_size=(64,128,128), positive_ratio=0.7, cache_size=3):
        """
        - df must have SeriesInstanceUID.
        - Loads/synthesizes masks on the fly. Keeps a small LRU cache of volumes.
        - positive_ratio: probability to sample a patch that contains ANY foreground class (>0).
        """
        self.df = df.copy().reset_index(drop=True)
        if n_patients:
            self.df = self.df.head(n_patients)
        self.series_uids = self.df["SeriesInstanceUID"].tolist()
        self.patch_size = patch_size
        self.positive_ratio = positive_ratio
        self.cache = {}
        self.cache_order = []
        self.cache_size = cache_size

    def __len__(self):
        # Nominal number of patches per epoch
        return len(self.series_uids) * 20

    def _get_from_cache(self, uid):
        if uid in self.cache:
            return self.cache[uid]
        # Load volume
        mask_path = find_mask_path(uid)
        if mask_path and os.path.exists(mask_path):
            mask = load_nifti_volume(mask_path).astype(np.uint8)
        else:
            mask = None

        # Prefer DICOM for image
        img = load_dicom_series(os.path.join(SERIES_FOLDER, uid)).astype(np.float32)

        if mask is None:
            mask = synthesize_mask_from_localizer(uid, img.shape)
        else:
            if mask.shape != img.shape:
                # up/downsample mask to match image (nearest)
                m = sk_resize(mask, img.shape, order=0, preserve_range=True, anti_aliasing=False)
                mask = (m + 0.5).astype(np.uint8)

        self.cache[uid] = (img, mask)
        self.cache_order.append(uid)
        if len(self.cache_order) > self.cache_size:
            old = self.cache_order.pop(0)
            self.cache.pop(old, None)
        return img, mask

    def __getitem__(self, idx):
        uid = random.choice(self.series_uids)
        img, mask = self._get_from_cache(uid)  # (D,H,W)
        D, H, W = img.shape
        pd, ph, pw = self.patch_size

        # choose z slice with foreground sometimes
        if random.random() < self.positive_ratio and mask.sum() > 0:
            slice_sums = mask.sum(axis=(1,2))
            pos = np.where(slice_sums > 0)[0]
            z_center = int(random.choice(pos)) if len(pos) else random.randint(0, D-1)
        else:
            z_center = random.randint(0, D-1)

        z0 = np.clip(z_center - pd//2, 0, max(0, D - pd))
        y0 = random.randint(0, max(0, H - ph)) if H > ph else 0
        x0 = random.randint(0, max(0, W - pw)) if W > pw else 0

        vol_patch = img[z0:z0+pd, y0:y0+ph, x0:x0+pw]
        msk_patch = mask[z0:z0+pd, y0:y0+ph, x0:x0+pw]

        # pad if edges
        if vol_patch.shape != (pd,ph,pw):
            p = np.zeros((pd,ph,pw), dtype=vol_patch.dtype)
            p[:vol_patch.shape[0], :vol_patch.shape[1], :vol_patch.shape[2]] = vol_patch
            vol_patch = p
        if msk_patch.shape != (pd,ph,pw):
            p = np.zeros((pd,ph,pw), dtype=msk_patch.dtype)
            p[:msk_patch.shape[0], :msk_patch.shape[1], :msk_patch.shape[2]] = msk_patch
            msk_patch = p

        # simple flips
        if random.random() < 0.5:
            vol_patch = np.flip(vol_patch, axis=1).copy()
            msk_patch = np.flip(msk_patch, axis=1).copy()
        if random.random() < 0.5:
            vol_patch = np.flip(vol_patch, axis=2).copy()
            msk_patch = np.flip(msk_patch, axis=2).copy()

        x = torch.from_numpy(vol_patch).unsqueeze(0).float()  # (1,D,H,W)
        y = torch.from_numpy(msk_patch).long()                # (D,H,W)
        return x, y


In [6]:
# Cell 4 - Model: Compact 3D U-Net

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, num_classes=NUM_CLASSES, base_ch=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base_ch)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = ConvBlock(base_ch, base_ch*2)
        self.pool2 = nn.MaxPool3d(2)
        self.enc3 = ConvBlock(base_ch*2, base_ch*4)
        self.pool3 = nn.MaxPool3d(2)
        self.bottleneck = ConvBlock(base_ch*4, base_ch*8)
        self.up3 = nn.ConvTranspose3d(base_ch*8, base_ch*4, 2, 2)
        self.dec3 = ConvBlock(base_ch*8, base_ch*4)
        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, 2, 2)
        self.dec2 = ConvBlock(base_ch*4, base_ch*2)
        self.up1 = nn.ConvTranspose3d(base_ch*2, base_ch, 2, 2)
        self.dec1 = ConvBlock(base_ch*2, base_ch)
        self.out = nn.Conv3d(base_ch, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2)
        e3 = self.enc3(p2); p3 = self.pool3(e3)
        b  = self.bottleneck(p3)
        u3 = self.up3(b);  d3 = self.dec3(torch.cat([u3, e3], dim=1))
        u2 = self.up2(d3); d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2); d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1)

model = UNet3D().to(device)
print("Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 3))


Params (M): 1.401


In [7]:
# Cell 5 - Losses & Metrics: Class-weighted CE + Dice (foreground)

def build_class_weights_from_localizers(localizer_df, label_cols, bg_weight=0.05, min_w=0.5, max_w=3.0):
    """Heuristic weights using localizer frequency per class."""
    counts = Counter()
    for _, r in localizer_df.iterrows():
        loc = r.get("location", "")
        if loc in label_cols:
            counts[loc] += 1
        else:
            counts["Other Posterior Circulation"] += 1
    weights = np.ones(len(label_cols) + 1, dtype=np.float32)
    weights[0] = bg_weight
    # inverse log frequency
    for name, idx in {n:i+1 for i,n in enumerate(label_cols)}.items():
        c = max(1, counts.get(name, 1))
        w = 1.0 / math.log(c + 2.0)
        weights[idx] = float(np.clip(w, min_w, max_w))
    return torch.tensor(weights, dtype=torch.float32)

class DiceLossMC(nn.Module):
    def __init__(self, ignore_index=0, smooth=1e-6):
        super().__init__()
        self.ignore_index = ignore_index
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        n, c, d, h, w = probs.shape
        targets_oh = F.one_hot(targets, c).permute(0,4,1,2,3).float()
        # exclude background
        mask = torch.ones(c, device=logits.device, dtype=torch.bool)
        mask[self.ignore_index] = False
        probs_f = probs[:, mask]
        target_f = targets_oh[:, mask]
        dims = (0,2,3,4)
        inter = (probs_f * target_f).sum(dim=dims)
        denom = probs_f.sum(dim=dims) + target_f.sum(dim=dims)
        dice = (2*inter + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()

class DiceCELoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLossMC(ignore_index=0)
    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)

@torch.no_grad()
def compute_metrics(logits, targets):
    probs = F.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
    acc = (preds == targets).float().mean().item()

    # per-class dice including bg for inspection
    c = logits.shape[1]
    targets_oh = F.one_hot(targets, c).permute(0,4,1,2,3).float()
    dims = (0,2,3,4)
    inter = (probs * targets_oh).sum(dim=dims)
    denom = probs.sum(dim=dims) + targets_oh.sum(dim=dims)
    dice = ((2*inter) / (denom + 1e-6)).cpu().numpy()  # (C,)
    # foreground mean (exclude bg)
    fg_dice = dice[1:].mean() if c > 1 else dice.mean()
    return float(fg_dice), float(acc), dice


In [8]:
# Cell 6 - Prepare DataFrames and DataLoaders

df = pd.read_csv(FILTERED_CSV)
df = df[df['SeriesInstanceUID'].notna()].reset_index(drop=True)
n_pat = min(N_PATIENTS, len(df))
df = df.head(n_pat).reset_index(drop=True)

# Simple split
n_train = int(0.9 * len(df))
train_df = df.iloc[:n_train].reset_index(drop=True)
val_df   = df.iloc[n_train:].reset_index(drop=True)

# Datasets
train_ds = VolumePatchDataset(train_df, n_patients=None, patch_size=PATCH_SIZE, positive_ratio=0.8, cache_size=2)
val_ds   = VolumePatchDataset(val_df,   n_patients=None, patch_size=PATCH_SIZE, positive_ratio=0.0, cache_size=2)

# Loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print(f"Train patients: {len(train_ds.series_uids)} | Val patients: {len(val_ds.series_uids)}")
print("Train batches/epoch:", len(train_loader), "| Val batches:", len(val_loader))


Train patients: 36 | Val patients: 4
Train batches/epoch: 360 | Val batches: 40


In [ ]:
# Cell 7 - Training loop with OOM safety & best checkpoint

# Class weights from localizers (fallback to uniform if file missing)
if len(localizer_df) == 0:
    class_weights = torch.tensor([0.05] + [1.0]*(NUM_CLASSES-1), dtype=torch.float32)
else:
    class_weights = build_class_weights_from_localizers(localizer_df, LABEL_COLS, bg_weight=0.05)
class_weights = class_weights.to(device)

criterion = DiceCELoss(class_weights=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)

train_loss_hist, val_loss_hist, val_dice_hist, val_acc_hist = [], [], [], []
best_dice = -1.0
cur_bs = BATCH_SIZE

def safe_train_step(data, target):
    global cur_bs
    try:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(data)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        return loss.item(), None
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            # Signal caller to drop batch or reduce bs externally
            return None, "oom"
        else:
            raise

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    running = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", leave=False)
    for xb, yb in pbar:
        loss_val, err = safe_train_step(xb, yb)
        if err == "oom":
            print("⚠️ OOM on this batch. Skipping after clearing cache.")
            continue
        if loss_val is not None:
            running.append(loss_val)
            pbar.set_postfix(loss=np.mean(running))
    tr_loss = float(np.mean(running)) if running else 0.0
    train_loss_hist.append(tr_loss)

    # ---- Validation ----
    model.eval()
    v_losses, v_dices, v_accs = [], [], []
    with torch.no_grad():
        for xb, yb in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]", leave=False):
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb).item()
            fg_dice, acc, _ = compute_metrics(logits, yb)
            v_losses.append(loss); v_dices.append(fg_dice); v_accs.append(acc)

    val_loss = float(np.mean(v_losses)) if v_losses else 0.0
    val_dice = float(np.mean(v_dices)) if v_dices else 0.0
    val_acc  = float(np.mean(v_accs)) if v_accs else 0.0

    val_loss_hist.append(val_loss); val_dice_hist.append(val_dice); val_acc_hist.append(val_acc)
    print(f"Epoch {epoch}: TrainLoss {tr_loss:.4f} | ValLoss {val_loss:.4f} | ValDice {val_dice:.4f} | ValAcc {val_acc:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        ckpt = os.path.join(OUTPUT_DIR, "best_unet3d_multiclass.pth")
        torch.save({"model": model.state_dict(), "epoch": epoch, "dice": best_dice}, ckpt)
        print("💾 Saved best model:", ckpt)


Epoch 1/3 [train]:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 1/3 [val]:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 1: TrainLoss 3.3424 | ValLoss 3.1043 | ValDice 0.0003 | ValAcc 0.8244
💾 Saved best model: /kaggle/working/patient_visuals/best_unet3d_multiclass.pth


Epoch 2/3 [train]:   0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
# Cell 8 - Inference utils: sliding window + color overlays + legend

def sliding_window_predict(model, vol, patch_size=(64,128,128), stride=(32,96,96)):
    """
    vol: (D,H,W) float32 in [0,1]
    returns pred mask (D,H,W) uint8 by tiling with overlap and averaging logits.
    """
    model.eval()
    pd, ph, pw = patch_size
    sd, sh, sw = stride
    D, H, W = vol.shape

    # pad to cover edges
    Pd = (math.ceil((D - pd) / sd) * sd + pd) if D > pd else pd
    Ph = (math.ceil((H - ph) / sh) * sh + ph) if H > ph else ph
    Pw = (math.ceil((W - pw) / sw) * sw + pw) if W > pw else pw

    pad_vol = np.zeros((Pd, Ph, Pw), dtype=vol.dtype)
    pad_vol[:D, :H, :W] = vol

    logit_sum = np.zeros((NUM_CLASSES, Pd, Ph, Pw), dtype=np.float32)
    count_sum = np.zeros((Pd, Ph, Pw), dtype=np.float32)

    with torch.no_grad():
        for z in range(0, Pd - pd + 1, sd):
            for y in range(0, Ph - ph + 1, sh):
                for x in range(0, Pw - pw + 1, sw):
                    patch = pad_vol[z:z+pd, y:y+ph, x:x+pw]
                    x_t = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).float().to(device)
                    logits = model(x_t)[0].cpu().numpy()  # (C,D,H,W)
                    logit_sum[:, z:z+pd, y:y+ph, x:x+pw] += logits
                    count_sum[z:z+pd, y:y+ph, x:x+pw] += 1.0

    logit_sum /= (count_sum[None] + 1e-6)
    pred = logit_sum.argmax(axis=0).astype(np.uint8)
    return pred[:D, :H, :W]

# Color map per class
import matplotlib.colors as mcolors
BASE_CMAP = plt.get_cmap("tab20")  # plenty of distinct colors
CLASS_COLORS = [(0,0,0,0.0)]  # transparent for background overlay
for i in range(len(LABEL_COLS)):
    CLASS_COLORS.append(BASE_CMAP(i % BASE_CMAP.N))
CLASS_ID2NAME = {0: "Background", **{i+1: n for i,n in enumerate(LABEL_COLS)}}

def mask_to_rgba(mask):
    """(H,W) -> (H,W,4) RGBA using CLASS_COLORS."""
    h, w = mask.shape
    out = np.zeros((h,w,4), dtype=np.float32)
    for cid, col in enumerate(CLASS_COLORS):
        out[mask == cid] = col
    return out

def visualize_patient_overlays(series_uid, model, save=True, max_slices=12):
    # Load volume + ground-truth (nii or synth)
    mp = find_mask_path(series_uid)
    if mp and os.path.exists(mp):
        gt = load_nifti_volume(mp).astype(np.uint8)
        vol = load_dicom_series(os.path.join(SERIES_FOLDER, series_uid)).astype(np.float32)
        if gt.shape != vol.shape:
            gt = (sk_resize(gt, vol.shape, order=0, preserve_range=True, anti_aliasing=False)+0.5).astype(np.uint8)
    else:
        vol = load_dicom_series(os.path.join(SERIES_FOLDER, series_uid)).astype(np.float32)
        gt = synthesize_mask_from_localizer(series_uid, vol.shape)

    # Predict
    pred = sliding_window_predict(model, vol, patch_size=PATCH_SIZE, stride=(PATCH_SIZE[0]//2, PATCH_SIZE[1]//2, PATCH_SIZE[2]//2))

    # choose slice indices
    D = vol.shape[0]
    idxs = np.linspace(0, D-1, num=min(max_slices, D), dtype=int)

    cols = 4
    rows = int(math.ceil(len(idxs)/cols))
    fig, axs = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axs = np.array(axs).reshape(-1)

    for ax, z in zip(axs, idxs):
        ax.imshow(vol[z], cmap="gray")
        ax.imshow(mask_to_rgba(gt[z]), alpha=0.35)
        ax.imshow(mask_to_rgba(pred[z]), alpha=0.35)
        ax.set_title(f"{series_uid} | z={z}")
        ax.axis('off')

    # Legend
    handles = [plt.Line2D([0],[0], marker='s', linestyle='None', color=CLASS_COLORS[i], label=CLASS_ID2NAME[i], markersize=10)
               for i in range(1, NUM_CLASSES)]
    if len(handles) > 0:
        fig.legend(handles=handles, loc='lower center', ncol=min(5, len(handles)), bbox_to_anchor=(0.5, 0.02))

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    if save:
        outp = os.path.join(OUTPUT_DIR, f"{series_uid}_overlay.png")
        plt.savefig(outp, dpi=120)
        plt.close()
        print("Saved overlay ->", outp)
    else:
        plt.show()


In [ ]:
# Cell 9 - Train

# (Re-run Cell 7 if you tweak model/loss)
# Training might take a while; start with NUM_EPOCHS=3 to smoke test, then increase.

# Already executed in Cell 7 loop block; if you want to re-train explicitly, re-run Cell 7.
# Nothing to execute here unless you reset and want a single-call wrapper.


In [ ]:
# Cell 10 - Visualize a few validation patients with class legend

val_uids = val_df["SeriesInstanceUID"].tolist()[:4]
for uid in val_uids:
    try:
        visualize_patient_overlays(uid, model, save=True, max_slices=12)
    except Exception as e:
        print("Visualization failed for", uid, "->", e)


In [ ]:
# Cell 11 - Plot training curves

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.plot(train_loss_hist, label="train"); plt.plot(val_loss_hist, label="val"); plt.title("Loss"); plt.legend()
plt.subplot(1,3,2); plt.plot(val_dice_hist, label="val_dice"); plt.title("Foreground Dice"); plt.legend()
plt.subplot(1,3,3); plt.plot(val_acc_hist, label="val_acc"); plt.title("Pixel Accuracy"); plt.legend()
plt.tight_layout(); plt.show()
